In [ ]:
# Automatically reloads all code changes in python modules when changes are saved
%load_ext autoreload
%autoreload 2

In [ ]:
## Initialize logger for whole project
## Also tracks output of YOLO ML training to this log file

from flower_classifier.logging import setup_logging
from ultralytics import YOLO

# LOG_FILE = 'flower_classifier.log'
# logger = setup_logging(log_file=str(LOG_FILE))
logger = setup_logging()
logger.debug(f"Logging setup successful.")

In [ ]:
## Sets up environment variables with kaggle
## MUST HAVE .env FILE WITH FILLED IN KAGGLE LOGIN INFO EVEN IF USE_KAGGLE IS FALSE
from dotenv import load_dotenv
from pathlib import Path
from flower_classifier.config_IO import PROJECT_ROOT
import os

file_path = PROJECT_ROOT / ".env"
if not os.path.isfile(file_path):
    logger.error("'.env' file not found in the project root. Please make .env file with KAGGLE_USERNAME and KAGGLE_KEY. See README.md for more information.")
else:
    logger.info(".env file found! Importing configs.")

# Extract environment variables from .env file
load_dotenv()
KAGGLE_USERNAME = os.getenv('KAGGLE_USERNAME')
KAGGLE_KEY = os.getenv('KAGGLE_KEY')
GOOGLE_ZIP_ID = os.getenv('GOOGLE_ZIP_ID')

# # Resolve project root directory
# path = Path(__name__).resolve().parent
# while path.name != "flower-classifier" and path != path.parent:
#     path = path.parent
# logger.info(f"Path to project parent directory: '{path}'")
# PROJECT_ROOT = path

In [ ]:
from flower_classifier.data_setup import download_dataset, unzip_dataset, run_ingestion
from flower_classifier.config_IO import save_json_config, load_json_config, make_YOLO26_config, get_highest_numbered_run
from flower_classifier.config_IO import print_system_info, verify_venv
from flower_classifier.file_io import view_image, view_text

# import get_dataset
# import ingest
# import config_IO
# import system_info
# import view_file

from pathlib import Path
import os
import sys
import platform
import json
import multiprocessing
import yaml
import torch

In [ ]:
try:
    verify_venv()
except Exception as e:
    logger.exception(f"Exception while verifying venv: \n{e}")
    raise e
    
logger.info("Venv verified successfully")

In [ ]:
USE_KAGGLE = False

zip_location = PROJECT_ROOT / 'data' / 'compressed' / 'tomato-flower-3-class.zip'

if not zip_location.is_file():
    logger.info(f"Zip file not previously found, beginning retrieval.")
    
    do_unzip = download_dataset(from_kaggle=USE_KAGGLE, kaggle_username=KAGGLE_USERNAME, kaggle_key=KAGGLE_KEY, google_zip_id=GOOGLE_ZIP_ID)
    
    unzip_dataset()
    
else:
    logger.info(f"zip file already exists at '{zip_location}', skipping download.")

In [ ]:
if not os.path.exists(str(PROJECT_ROOT / 'data/data_set/tomato-flower/val/images')) and not os.path.exists(str(PROJECT_ROOT / 'data/data_set/tomato-flower/test/images')):
    # if ingestion has not already been done, then run the ingestion script.
    logger.info("Ingestion not yet done. Beginning ingestion.")
    run_ingestion()
else:
    logger.info("Ingestion done previously. Skipping ingestion.")

logger.info("Ingestion step complete.")

In [ ]:
make_YOLO26_config()

In [ ]:
print_system_info()

In [ ]:
## Load parameters from specified file location

file_location =  str(PROJECT_ROOT / "config/YOLO26_config.json")
TRAIN_CONFIG = load_json_config(file_location)

In [ ]:
## Manual overrides

## for example:
# cfg["run_name"] = "tomato_flower_yolo26_trial_02"
# cfg["train_params"]["epochs"] = 150

## We change the configs within the 'cfg' object as opposed to directly on MODEL_CFG or other imported vars.
## This is so, whenever we save the configs, they can be exported back to the JSON config file if we wish.

In [ ]:
## CPU/GPU Usage Config

CPU_COUNT = multiprocessing.cpu_count()
DATALOADER_WORKERS = TRAIN_CONFIG["train_params"]["workers"]
CPU_THREADS_PER_PROC = max(1, CPU_COUNT // max(1, DATALOADER_WORKERS))

os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)

torch.set_num_threads(CPU_THREADS_PER_PROC)

print(f"CPU count: {CPU_COUNT}")
print(f"DataLoader workers: {DATALOADER_WORKERS}")
print(f"CPU threads per process: {CPU_THREADS_PER_PROC}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
## PREFLIGHT CHECKS
## Checks ALL system configs for current settings.
## Also catches many common setup errors or issues.


# ==========================
# USER CONFIG (from your setup)
# ==========================
DATA_YAML = PROJECT_ROOT / "config/data.yaml"
DATASET_ROOT = PROJECT_ROOT / "data/data_set/tomato-flower"
PROJECT_DIR = PROJECT_ROOT / "runs"
RUN_NAME = "tomato_flower_yolo26_from_scratch"
MODEL_CFG = "yolo26n.yaml"

TRAIN_PARAMS = TRAIN_CONFIG["train_params"]

# ==========================
# CPU THREAD CONTROL (GPU TRAINING SAFE)
# ==========================
CPU_COUNT = multiprocessing.cpu_count()
WORKERS = TRAIN_PARAMS.get("workers", 8)

CPU_THREADS_PER_PROC = max(1, CPU_COUNT // max(1, WORKERS))

os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS_PER_PROC)

torch.set_num_threads(CPU_THREADS_PER_PROC)

logger.info(
    f"\n=== CPU THREAD CONFIG ==="
    f"\n CPU count              : {CPU_COUNT}"
    f"\n DataLoader workers     : {WORKERS}"
    f"\n Threads per process    : {CPU_THREADS_PER_PROC}"
    f"\n torch num threads      : {torch.get_num_threads()}"
    f"\n"
)

# ==========================
# SYSTEM INFO
# ==========================
logger.info(
    f"\n=== SYSTEM INFO ==="
    f"\n Python executable : {sys.executable}"
    f"\n Python version    : {sys.version.splitlines()[0]}"
    f"\n Platform          : {platform.platform()}"
    f"\n Machine           : {platform.machine()}"
    f"\n Processor         : {platform.processor()}"
    f"\n CPU count         : {CPU_COUNT}"
    f"\n Working dir       : {Path.cwd()}"
    f"\n"
)

# ==========================
# TRAIN CONFIG SUMMARY
# ==========================
logger.info(
    f"\n=== TRAIN CONFIG SUMMARY ==="
    f"\n model_cfg   : {TRAIN_CONFIG['model_cfg']}"
    f"\n data_yaml   : {TRAIN_CONFIG['data_yaml']}"
    f"\n project_dir : {TRAIN_CONFIG['project_dir']}"
    f"\n run_name    : {TRAIN_CONFIG['run_name']}"
    f"\n epochs      : {TRAIN_PARAMS['epochs']}"
    f"\n imgsz       : {TRAIN_PARAMS['imgsz']}"
    f"\n batch       : {TRAIN_PARAMS['batch']}"
    f"\n device      : {TRAIN_PARAMS['device']}"
    f"\n workers     : {TRAIN_PARAMS['workers']}"
    f"\n amp         : {TRAIN_PARAMS['amp']}"
    f"\n optimizer   : {TRAIN_PARAMS['optimizer']}"
    f"\n pretrained  : {TRAIN_PARAMS['pretrained']}"
    f"\n"
)

# ==========================
# TORCH / CUDA INFO
# ==========================
cuda_available = torch.cuda.is_available()
mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

logger.info(
    f"\n=== TORCH INFO ==="
    f"\n torch version       : {torch.__version__}"
    f"\n CUDA available      : {cuda_available}"
    f"\n CUDA device count   : {torch.cuda.device_count() if cuda_available else 0}"
    f"\n MPS available       : {mps_available}"
    f"\n"
)

if cuda_available:
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        total_mem_gb = props.total_memory / (1024 ** 3)

        logger.info(
            f"\nCUDA DEVICE {i}:"
            f"\n name              = {torch.cuda.get_device_name(i)}"
            f"\n capability        = {props.major}.{props.minor}"
            f"\n total_memory_gb   = {total_mem_gb:.2f}"
            f"\n multiprocessors   = {props.multi_processor_count}"
        )
else:
    logger.warning("CUDA is NOT available. Training will fall back to CPU.")

# ==========================
# CUDA SMOKE TEST
# ==========================
logger.info(f"Testing CUDA usage...")
if cuda_available:
    try:
        x = torch.tensor([1.0, 2.0, 3.0], device="cuda")
        logger.info(f"CUDA smoke test PASSED (tensor device = {x.device})")
        del x
        torch.cuda.empty_cache()
    except Exception as e:
        logger.exception(f"CUDA detected but failed runtime test: {e}")
        raise RuntimeError(f"CUDA detected but failed runtime test: {e}")

# ==========================
# ENV VARS
# ==========================
cuda_visible_devices = os.environ.get('CUDA_VISIBLE_DEVICES')
logger.info(
    f"\n=== IMPORTANT ENV VARS ==="
    f"\n OMP_NUM_THREADS        = {os.environ.get('OMP_NUM_THREADS')}"
    f"\n MKL_NUM_THREADS        = {os.environ.get('MKL_NUM_THREADS')}"
    f"\n NUMEXPR_NUM_THREADS    = {os.environ.get('NUMEXPR_NUM_THREADS')}"
    f"\n OPENBLAS_NUM_THREADS   = {os.environ.get('OPENBLAS_NUM_THREADS')}"
    f"\n CUDA_VISIBLE_DEVICES   = {cuda_visible_devices if cuda_visible_devices is not None else '<Not restricted>'}"
    f"\n ULTRALYTICS_CONFIG_DIR = {os.environ.get('ULTRALYTICS_CONFIG_DIR')}"
    f"\n"
)

if os.environ.get("CUDA_VISIBLE_DEVICES") is not None:
    logger.info(
        f"CUDA_VISIBLE_DEVICES is set to {os.environ.get('CUDA_VISIBLE_DEVICES')} "
        "→ This may restrict visible GPUs."
    )

# ==========================
# DEVICE VALIDATION
# ==========================
requested_device = TRAIN_PARAMS["device"]

if requested_device != "cpu" and not cuda_available:
    raise RuntimeError(
        f"Config requests GPU (device={requested_device}) but CUDA is unavailable."
    )

if requested_device == -1:
    logger.info("device=-1 → auto-selecting most idle GPU.")

# ==========================
# HELPERS
# ==========================
def count_files(path, exts=None):
    if not path.exists():
        return 0
    files = [p for p in path.iterdir() if p.is_file()]
    if exts:
        exts = {e.lower() for e in exts}
        files = [p for p in files if p.suffix.lower() in exts]
    return len(files)

def print_path_status(label, path):
    path = Path(path)
    logger.info(
        f"\n{label}:"
        f"\n path    = {path}"
        f"\n exists  = {path.exists()}"
        f"\n is_dir  = {path.is_dir()}"
        f"\n is_file = {path.is_file()}"
    )

# ==========================
# PATH CHECKS
# ==========================
logger.info("\n=== PATH CHECKS ===")

print_path_status("DATA_YAML", DATA_YAML)
print_path_status("DATASET_ROOT", DATASET_ROOT)
print_path_status("PROJECT_DIR", PROJECT_DIR)

for split in ["train", "val", "test"]:
    split_dir = DATASET_ROOT / split
    img_dir = split_dir / "images"
    lbl_dir = split_dir / "labels"

    print_path_status(f"{split.upper()} DIR", split_dir)

    img_count = count_files(img_dir, {".jpg", ".jpeg", ".png", ".bmp", ".webp"})
    lbl_count = count_files(lbl_dir, {".txt"})

    logger.info(
        f"{split.upper()} COUNTS:"
        f"\n image files = {img_count}"
        f"\n label files = {lbl_count}"
    )

# ==========================
# YAML VALIDATION
# ==========================
logger.info("\n=== YAML CONTENTS ===")

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

logger.info(json.dumps(data_cfg, indent=2))

logger.info("\n=== YAML PATH VALIDATION ===")

for key in ["train", "val", "test"]:
    p = Path(data_cfg[key])
    logger.info(
        f"{key}: {p}"
        f"\n exists = {p.exists()}"
        f"\n is_dir = {p.is_dir()}"
    )

logger.info("\n=== PREFLIGHT CHECK COMPLETE ===\n")

In [ ]:
## Verify YOLO26 is accessible and usable

try:
    YOLO("yolo26n.yaml")
    logger.info("DONE! Model config recognized")
except Exception as e:
    logger.info("ERROR! Model config NOT recognized")
    logger.info(e)

In [ ]:
## Get current run number from the file names

RUN_NUM = get_highest_numbered_run(TRAIN_CONFIG["project_dir"])
logger.info(f"Highest previous run found is {RUN_NUM}.")

In [ ]:
## Training the YOLO26 model

# Run number is automatically incremented based on the folders that exists in the output directory
# This is due to 'exist_ok=False' instead of True, where it would just replace the dir instead of making an incremented new one. 
RUN_NUM += 1

try:
    logger.info(f"Beginning training run {RUN_NUM}")
    
    model = YOLO(TRAIN_CONFIG["model_cfg"])
    run_name = str(TRAIN_CONFIG["run_name"]) + RUN_NUM

    results = model.train(
        data=TRAIN_CONFIG["data_yaml"],
        project=TRAIN_CONFIG["project_dir"],
        name=run_name,
        **TRAIN_CONFIG["train_params"]
    )

    logger.info(f"DONE! Training run {RUN_NUM} complete!")

    
except Exception as e:
    logger.exception(f"ERROR while training model: \n{e}")


In [ ]:
## Save current config values back to specified config json file.
CONFIG_PATH = str(PROJECT_ROOT / f"runs/exp_{RUN_NUM}/configs_used.json")
logger.info(f"Saving configs for run {RUN_NUM} to '{CONFIG_PATH}'")

save_json_config(TRAIN_CONFIG, CONFIG_PATH)
logger.info("Configs saved successfully!")

logger.info("Training complete!")